# Baseline Model Evaluation - StyleFit AI

**Phase**: Baseline Models  
**Dataset**: Rent the Runway (`data/raw/renttherunway_final_data.json.gz`)  
**Target**: Multi-class clothing fit (`fit`, `small`, `large`)  
**Product Goal**: Predict clothing fit likelihood *prior to purchase/rental* to reduce returns and improve customer sizing confidence.

---

## Executive Summary & Baseline Objectives

Before deploying complex tree ensembles or deep learning architectures, we must establish **honest, reproducible baseline benchmarks**.

### Objectives of this Phase:
1. **Majority-Class Baseline**: Evaluate `DummyClassifier(strategy='most_frequent')` to demonstrate why raw Accuracy is a highly misleading metric on imbalanced targets.
2. **Linear Baseline Models**: Fit `Multinomial Logistic Regression` with unweighted loss (`class_weight=None`) comparing two clothing size strategies:
   - **Numeric Size Strategy**: `size` treated as a scaled numeric feature.
   - **Categorical Size Strategy**: `size` treated as a categorical string feature with one-hot encoding.
3. **Dual Evaluation Framework**:
   - **Stratified Random Benchmark**: 80/20 train/test split preserving target class ratios.
   - **New-User Generalization Benchmark**: Group-aware split (`StratifiedGroupKFold`) ensuring zero `user_id` overlap to evaluate generalization to first-time customers.
4. **Strict Leakage Prevention**: Feature selection strictly enforces `GENERAL_FIT_FEATURES`. Post-purchase fields (`rating`, `review_text`, `review_summary`) and metadata identifiers (`user_id`, `item_id`, `review_date`) are strictly excluded.


In [1]:
# Environment setup & module imports
import os
import sys
from pathlib import Path

# Ensure project root is in Python path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.cleaning import clean_dataset, remove_exact_duplicates
from src.preprocessing import select_general_fit_features, build_preprocessor
from src.splitting import stratified_random_split, unseen_user_split, verify_split_integrity
from src.evaluation import evaluate_baseline_model, build_summary_dataframe, CLASS_ORDER
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# Display configuration
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
print('Imports initialized successfully. CLASS_ORDER:', CLASS_ORDER)


Imports initialized successfully. CLASS_ORDER: ['small', 'fit', 'large']


## 1. Data Ingestion, Deterministic Cleaning & Feature Selection

We load the raw JSON dataset, apply stateless deterministic cleaning, remove exact full-row redundant duplicates, and enforce our explicit pre-purchase allowlist boundary.

In [2]:
data_path = project_root / 'data' / 'raw' / 'renttherunway_final_data.json.gz'
print(f'Loading raw dataset from {data_path}...')
df_raw = pd.read_json(data_path, lines=True)
print(f'Raw dataset shape: {df_raw.shape}')

# Step 1: Clean dataset
df_clean = clean_dataset(df_raw)

# Step 2: Remove exact duplicates
df_dedup = remove_exact_duplicates(df_clean)
print(f'Deduplicated dataset shape: {df_dedup.shape}')

# Step 3: Enforce strict General Fit Model feature boundary
X_all = select_general_fit_features(df_dedup)
y_all = df_dedup['fit'].copy()
user_ids_all = df_dedup['user_id'].copy()

print(f'Selected pre-purchase features ({X_all.shape[1]} columns):')
print(list(X_all.columns))
print('\nTarget distribution:')
print(y_all.value_counts(normalize=True).to_frame('Proportion').round(4))


Loading raw dataset from C:\Users\mahta\my_projects\stylefit-ai\data\raw\renttherunway_final_data.json.gz...


Raw dataset shape: (192544, 15)


Deduplicated dataset shape: (192355, 19)
Selected pre-purchase features (9 columns):
['parsed_height_inches', 'parsed_weight_lbs', 'bust_band_size', 'bust_cup_size', 'body_type', 'age', 'size', 'category', 'rented_for']

Target distribution:
       Proportion
fit              
fit        0.7377
small      0.1339
large      0.1284


## 2. Dataset Splitting & Benchmark Setups

We set up two distinct evaluation benchmarks:
- **Setting A (Stratified Random Split)**: 80/20 random split stratified by target `fit`.
- **Setting B (Unseen-User Split)**: Group-aware split using candidate fold selection from `StratifiedGroupKFold(n_splits=5)`. We evaluate all candidate folds on size deviation (~20%) and target class distribution deviation, selecting the optimal fold strictly without invoking model training.

In [3]:
# Prepare DataFrame containing user_id and target for group-splitting
df_split_ready = X_all.copy()
df_split_ready['fit'] = y_all
df_split_ready['user_id'] = user_ids_all

# Setting A: Stratified Random Split
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = stratified_random_split(
    df_split_ready, target_col='fit', test_size=0.20, random_state=42
)
X_tr_rand = X_tr_rand.drop(columns=['user_id'])
X_te_rand = X_te_rand.drop(columns=['user_id'])

# Setting B: Unseen-User Split (Candidate Fold Selection)
X_tr_user, X_te_user, y_tr_user, y_te_user, u_tr_user, u_te_user = unseen_user_split(
    df_split_ready, user_col='user_id', target_col='fit', n_splits=5, random_state=42
)
X_tr_user = X_tr_user.drop(columns=['user_id'])
X_te_user = X_te_user.drop(columns=['user_id'])

# Integrity Verifications
rand_integrity = verify_split_integrity(X_tr_rand, X_te_rand, y_tr_rand, y_te_rand)
user_integrity = verify_split_integrity(X_tr_user, X_te_user, y_tr_user, y_te_user, u_tr_user, u_te_user)

print('=== SPLIT INTEGRITY RESULTS ===')
print('Stratified Random Split:', rand_integrity)
print('Unseen-User Group Split:', user_integrity)
print(f'Unseen-User Zero Overlap Check: {len(set(u_tr_user).intersection(set(u_te_user)))} overlapping users')


=== SPLIT INTEGRITY RESULTS ===
Stratified Random Split: {'index_disjoint': True, 'row_counts_match': True}
Unseen-User Group Split: {'index_disjoint': True, 'row_counts_match': True, 'zero_user_overlap': True}
Unseen-User Zero Overlap Check: 0 overlapping users


## 3. Baseline Model Fits & Evaluation

We execute all 3 baseline model configurations across both evaluation settings:
1. **DummyClassifier** (`most_frequent`)
2. **Multinomial Logistic Regression** (`size_strategy='numeric'`, `scale_numeric=True`)
3. **Multinomial Logistic Regression** (`size_strategy='categorical'`, `scale_numeric=True`)

Preprocessors are fitted **only** on training splits.

In [4]:
results = []

# Setting A: Stratified Random Split Evaluations
print('Evaluating Setting A (Stratified Random Split)...')
res_dummy_rand = evaluate_baseline_model(
    DummyClassifier(strategy='most_frequent'),
    X_tr_rand, y_tr_rand, X_te_rand, y_te_rand,
    'Dummy (Most Frequent)', 'Stratified Random'
)
results.append(res_dummy_rand)

res_lr_num_rand = evaluate_baseline_model(
    LogisticRegression(solver='lbfgs', class_weight=None, max_iter=1000, random_state=42),
    X_tr_rand, y_tr_rand, X_te_rand, y_te_rand,
    'Logistic Regression (Numeric Size)', 'Stratified Random',
    preprocessor=build_preprocessor(scale_numeric=True, size_strategy='numeric', min_frequency=100)
)
results.append(res_lr_num_rand)

res_lr_cat_rand = evaluate_baseline_model(
    LogisticRegression(solver='lbfgs', class_weight=None, max_iter=1000, random_state=42),
    X_tr_rand, y_tr_rand, X_te_rand, y_te_rand,
    'Logistic Regression (Categorical Size)', 'Stratified Random',
    preprocessor=build_preprocessor(scale_numeric=True, size_strategy='categorical', min_frequency=100)
)
results.append(res_lr_cat_rand)

# Setting B: Unseen-User Split Evaluations
print('Evaluating Setting B (Unseen-User Split)...')
res_dummy_user = evaluate_baseline_model(
    DummyClassifier(strategy='most_frequent'),
    X_tr_user, y_tr_user, X_te_user, y_te_user,
    'Dummy (Most Frequent)', 'Unseen User'
)
results.append(res_dummy_user)

res_lr_num_user = evaluate_baseline_model(
    LogisticRegression(solver='lbfgs', class_weight=None, max_iter=1000, random_state=42),
    X_tr_user, y_tr_user, X_te_user, y_te_user,
    'Logistic Regression (Numeric Size)', 'Unseen User',
    preprocessor=build_preprocessor(scale_numeric=True, size_strategy='numeric', min_frequency=100)
)
results.append(res_lr_num_user)

res_lr_cat_user = evaluate_baseline_model(
    LogisticRegression(solver='lbfgs', class_weight=None, max_iter=1000, random_state=42),
    X_tr_user, y_tr_user, X_te_user, y_te_user,
    'Logistic Regression (Categorical Size)', 'Unseen User',
    preprocessor=build_preprocessor(scale_numeric=True, size_strategy='categorical', min_frequency=100)
)
results.append(res_lr_cat_user)

# Build Consolidated Summary DataFrame
df_summary = build_summary_dataframe(results)
print('\n=== CONSOLIDATED BASELINE METRICS TABLE ===')
display(df_summary)


Evaluating Setting A (Stratified Random Split)...


Evaluating Setting B (Unseen-User Split)...



=== CONSOLIDATED BASELINE METRICS TABLE ===


,Model,Benchmark Split,Accuracy,Balanced Accuracy,Macro F1,Weighted F1,Macro ROC-AUC,Macro PR-AUC,F1 (small),F1 (fit),F1 (large),Recall (small),Recall (fit),Recall (large)
0,Dummy (Most Frequent),Stratified Random,0.737751,0.333333,0.283029,0.626414,0.500000,0.333333,0.000000,0.849087,0.000000,0.000000,1.000000,0.000000
1,Logistic Regression (Numeric Size),Stratified Random,0.736607,0.333996,0.285627,0.627013,0.613160,0.395431,0.006491,0.848369,0.002020,0.003302,0.997675,0.001012
2,Logistic Regression (Categorical Size),Stratified Random,0.737387,0.333994,0.284926,0.626981,0.627964,0.407763,0.001549,0.848803,0.004426,0.000777,0.998978,0.002227
3,Dummy (Most Frequent),Unseen User,0.737751,0.333333,0.283029,0.626414,0.500000,0.333333,0.000000,0.849087,0.000000,0.000000,1.000000,0.000000
4,Logistic Regression (Numeric Size),Unseen User,0.736373,0.334445,0.286876,0.627350,0.612797,0.393386,0.006861,0.848130,0.005637,0.003496,0.997005,0.002834
5,Logistic Regression (Categorical Size),Unseen User,0.737803,0.335182,0.287389,0.628058,0.627354,0.405475,0.001936,0.849010,0.011222,0.000971,0.998908,0.005668


## 4. Visualizations & Error Analysis

Below are the 5 saved baseline figures capturing model performance comparisons, confusion matrices, per-class metrics, and probability curves.

### Figure 01: Baseline Metric Comparison Across Evaluation Settings
![Figure 01: Overall Metrics Comparison](../reports/figures/baselines/01_baseline_overall_metrics_comparison.png)

---

### Figure 02: Dummy Classifier (Most Frequent) Normalized Confusion Matrix
![Figure 02: Dummy Confusion Matrix](../reports/figures/baselines/02_dummy_confusion_matrix.png)

---

### Figure 03: Logistic Regression Confusion Matrices (Numeric vs. Categorical Size)
![Figure 03: Logistic Regression Confusion Matrices Side-by-Side](../reports/figures/baselines/03_logistic_regression_confusion_matrices.png)

---

### Figure 04: Per-Class Precision, Recall, and F1-Score Comparison
![Figure 04: Per-Class Precision Recall F1](../reports/figures/baselines/04_per_class_precision_recall_f1.png)

---

### Figure 05: One-vs-Rest ROC & Precision-Recall Curves (Numeric vs. Categorical Logistic Regression)
![Figure 05: Logistic Regression Probability Curves](../reports/figures/baselines/05_logistic_regression_probability_curves.png)


## 5. Comprehensive Baseline Interpretation & Analysis

### Core Findings & Interpretation:

1. **Strong Majority Class Bias**: Both unweighted Logistic Regression baseline configurations (`size_strategy='numeric'` and `size_strategy='categorical'`) remain heavily biased toward the majority `fit` class (~73.8%). Under default 0.5 decision thresholds, unweighted cross-entropy loss predicts `fit` for over 99.4% of all test samples.
2. **Misleading Accuracy**: Accuracy near 74% (73.78%) is highly misleading because minority-class recall is near zero (Recall for `small` <= 0.35%, Recall for `large` <= 0.57%). High overall accuracy simply reflects predicting the majority class.
3. **Ranking Signal via ROC-AUC & PR-AUC**: Despite near-zero argmax recall, the ROC-AUC scores (~0.613 Numeric, ~0.627 Categorical) and PR-AUC scores (~0.393 Numeric, ~0.405 Categorical) demonstrate that meaningful continuous probability ranking signal exists for `small` and `large`. The models rank probability scores correctly far better than random chance, but default decision thresholds collapse predictions to the majority class.
4. **Motivation for Next Modeling Phase**: These findings directly motivate class-imbalance mitigation (e.g. `class_weight='balanced'`, decision threshold tuning) and stronger non-linear models (Random Forest, XGBoost, CatBoost) in the upcoming model-comparison phase.
5. **No Production Model Selected**: No production model has been selected in this baseline phase. These holdout evaluations serve strictly as comparative benchmarks for future modeling phases.

### Detailed Baseline Questions & Answers:

#### 1. What accuracy does the majority-class Dummy model achieve?
- The `DummyClassifier(strategy='most_frequent')` achieves an **Accuracy of 73.78%** on both the Stratified Random split and the Unseen-User split.

#### 2. Why is that accuracy misleading?
- The raw Accuracy of 73.78% is completely misleading because the model blindly predicts `fit` for 100% of inputs.
- For minority classes (`small` and `large`), **Recall is exactly 0.0%** and **F1-Score is 0.0**.
- Its **Balanced Accuracy is 33.33%** (random chance baseline for a 3-class target) and its **Macro F1 is 0.2830**.

#### 3. Does Logistic Regression improve Macro F1?
- Logistic Regression without class weighting (`class_weight=None`) achieves a **Macro F1 of ~0.2869 (Numeric Size)** and **~0.2874 (Categorical Size)**, representing only a negligible +0.0044 improvement over the Dummy baseline.

#### 4. Does it improve recall for small and large?
- Recall for `small` remains near zero (**0.35% for Numeric, 0.10% for Categorical**).
- Recall for `large` remains near zero (**0.28% for Numeric, 0.57% for Categorical**).

#### 5. Is numeric or categorical treatment of size better?
- **Categorical size treatment** demonstrates superior ranking and discrimination capability:
  - **Macro ROC-AUC**: 0.6274 (Categorical) vs. 0.6128 (Numeric)
  - **Macro PR-AUC**: 0.4055 (Categorical) vs. 0.3934 (Numeric)
- This indicates that clothing sizes carry discrete, non-linear categorical effects across body types that standard linear scaling compresses.

#### 6. How much performance changes when moving from random split to unseen-user evaluation?
- Macro F1 and Accuracy remain almost identical between Stratified Random split (0.2849) and Unseen-User group split (0.2874).
- This confirms that baseline underperformance stems from linear model capacity and unweighted class imbalance rather than user-level memorization or overfitting.

#### 7. What do the confusion matrices tell us about minority-class errors?
- The side-by-side confusion matrices in **Figure 03** show that **over 99.4% of actual small and large instances are misclassified as fit**.

#### 8. Why is PR-AUC (Precision-Recall AUC) critical under class imbalance?
- While ROC-AUC measures True Positive Rate vs False Positive Rate (which can be optimistic when the negative majority class is large), **PR-AUC directly measures Precision vs Recall for the minority positive class**.
- The baseline PR-AUC for `small` (~0.21) and `large` (~0.19) provides an unvarnished benchmark for minority-class recovery in future modeling phases.

---

## 6. Baseline Holdout Disclaimer & Next Steps

> [!IMPORTANT]
> **Baseline Holdout Disclaimer**:
> *These holdouts are used for comparative baseline benchmarking. Final model selection and final production evaluation will use a stricter validation strategy in the later model-comparison phase.*

### Key Takeaways for the Advanced Models Phase:
1. **Enforce Class Weighting**: Linear and tree models must incorporate `class_weight='balanced'` or target resampling to address the 73.8% majority class skew.
2. **Capture Non-Linear Feature Interactions**: Explore non-linear algorithms (Random Forest, XGBoost, LightGBM, CatBoost) capable of learning interactions between body metrics (`height`, `weight`, `bust_band_size`, `bust_cup_size`) and garment size.
3. **Threshold Tuning**: Move beyond default 0.5 decision thresholds by optimizing decision thresholds directly for Macro F1 and PR-AUC.
